# 데이터 업로드

In [3]:
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()

# 업로드한 파일 경로
df = pd.read_csv("train_target_only_cleaned.csv")

print(df.shape)
df.head()

Saving train_target_only_cleaned.csv to train_target_only_cleaned.csv
(1425, 7)


,시점,품목명,품종명,거래단위,등급,평년 평균가격(원),평균가격(원)
0,201801상순,건고추,화건,30 kg,상품,381666.666667,590000.0
1,201801중순,건고추,화건,30 kg,상품,380809.666667,590000.0
2,201801하순,건고추,화건,30 kg,상품,380000.000000,590000.0
3,201802상순,건고추,화건,30 kg,상품,380000.000000,590000.0
4,201802중순,건고추,화건,30 kg,상품,376666.666667,590000.0


# 최종 피처 구성

In [4]:
# ===============================
# 옵션
# ===============================
USE_LOG = False   # True로 바꾸면 로그 변환 버전 사용

# ===============================
# 기본 정리
# ===============================
df = df.copy()

# 시점 분해
df['year'] = df['시점'].str[:4].astype(int)
df['month'] = df['시점'].str[4:6].astype(int)
df['순'] = df['시점'].str[6:]  # 상순/중순/하순

# 순 → 숫자 변환
mapping = {'상순': 0, '중순': 1, '하순': 2}
df['순_num'] = df['순'].map(mapping)

# 시간 인덱스
df = df.sort_values(['품목명','품종명','거래단위','등급','year','month','순_num'])
df['time_idx'] = df.groupby(['품목명','품종명','거래단위','등급']).cumcount()

# ===============================
# 타겟 정의
# ===============================
target = '평균가격(원)'

if USE_LOG:
    df['y'] = np.log1p(df[target])
else:
    df['y'] = df[target]

# ===============================
# lag 피처
# ===============================
for lag in [1,2,3]:
    df[f'lag{lag}'] = df.groupby(['품목명','품종명','거래단위','등급'])['y'].shift(lag)

# ===============================
# ratio 피처 (diff 대신)
# ===============================
df['ratio1'] = df['lag1'] / (df['lag2'] + 1e-6)
df['ratio2'] = df['lag2'] / (df['lag3'] + 1e-6)

# ===============================
# rolling (최소만)
# ===============================
df['rolling_mean_3'] = df.groupby(['품목명','품종명','거래단위','등급'])['y'] \
                        .transform(lambda x: x.shift(1).rolling(3).mean())

# ===============================
# 계절성
# ===============================
df['sin_month'] = np.sin(2*np.pi*df['month']/12)
df['cos_month'] = np.cos(2*np.pi*df['month']/12)

# ===============================
# 평년 대비 (핵심)
# ===============================
df['normal_ratio'] = (df['평균가격(원)'] - df['평년 평균가격(원)']) / (df['평년 평균가격(원)'] + 1e-6)

# ===============================
# cross (사과-배)
# ===============================
pivot = df.pivot_table(
    index=['시점'],
    columns='품목명',
    values='y'
)

if '사과' in pivot.columns and '배' in pivot.columns:
    pivot['apple_pear_ratio'] = pivot['사과'] / (pivot['배'] + 1e-6)
    df = df.merge(pivot[['apple_pear_ratio']], on='시점', how='left')

# ===============================
# 최종 정리
# ===============================
df = df.dropna().reset_index(drop=True)

print("최종 shape:", df.shape)
df.head()

최종 shape: (1392, 23)


,시점,품목명,품종명,거래단위,등급,평년 평균가격(원),평균가격(원),year,month,순,...,lag1,lag2,lag3,ratio1,ratio2,rolling_mean_3,sin_month,cos_month,normal_ratio,apple_pear_ratio
0,201802상순,감자,감자 수미,20키로상자,상,28703.875000,55380.666667,2018,2,상순,...,50243.000000,48283.777778,44170.285714,1.040577,1.093128,47565.687831,0.866025,5.000000e-01,0.929379,0.731441
1,201802중순,감자,감자 수미,20키로상자,상,27419.882275,59133.000000,2018,2,중순,...,55380.666667,50243.000000,48283.777778,1.102256,1.040577,51302.481481,0.866025,5.000000e-01,1.156574,0.769739
2,201802하순,감자,감자 수미,20키로상자,상,29378.666667,60486.714286,2018,2,하순,...,59133.000000,55380.666667,50243.000000,1.067755,1.102256,54918.888889,0.866025,5.000000e-01,1.058865,0.763091
3,201803상순,감자,감자 수미,20키로상자,상,29317.527778,64069.777778,2018,3,상순,...,60486.714286,59133.000000,55380.666667,1.022893,1.067755,58333.460317,1.000000,6.123234e-17,1.185375,0.778125
4,201803중순,감자,감자 수미,20키로상자,상,30453.694444,68452.445000,2018,3,중순,...,64069.777778,60486.714286,59133.000000,1.059237,1.022893,61229.830688,1.000000,6.123234e-17,1.247755,0.751371


In [5]:
# ===============================
# 사용 피처 목록
# ===============================

feature_cols = [
    'lag1','lag2','lag3',
    'ratio1','ratio2',
    'rolling_mean_3',
    'sin_month','cos_month',
    '순_num',
    'time_idx',
    'normal_ratio',
    'apple_pear_ratio'
]

target_col = 'y'

print("사용 feature 개수:", len(feature_cols))
print(feature_cols)

사용 feature 개수: 12
['lag1', 'lag2', 'lag3', 'ratio1', 'ratio2', 'rolling_mean_3', 'sin_month', 'cos_month', '순_num', 'time_idx', 'normal_ratio', 'apple_pear_ratio']


[최종 피처 구성]

1. 시계열 기본 피처
- lag1, lag2, lag3  
  : 최근 3개 시점 가격

2. 변화율 피처
- ratio1 = lag1 / lag2  
- ratio2 = lag2 / lag3  

  : 가격 변화 비율 (diff 대신 사용)

  : 스케일 영향 줄이기 위함

3. rolling 피처
- rolling_mean_3  
  : 최근 3개 시점 평균 가격 (노이즈 완화)

4. 계절성 / 시점 피처
- month  
- sin_month, cos_month  
- 순_num (상순/중순/하순)  
  : 순 단위 패턴 반영
  - EDA 결과 차이는 별로 없었으나 예측해야하는 값이 순 단위라 넣었습니다

- time_idx  
  : 전체 시계열 흐름 반영  
  (2018–2021 → 2022 예측 구조 고려)

5. 평년가격 기반 피처
- 평년 평균가격(원)  
- normal_ratio = (현재가격 - 평년가격) / 평년가격  

  : 평년 대비 상대적 가격 수준 반영  
  (품목별 스케일 차이 보정)

6. cross feature (품목 간 관계)
- apple_pear_ratio = 사과 가격 / 배 가격  

  : 사과-배 간 대체/연관 관계 반영


[제외한 피처]
- source_type, is_imputed

  : 타겟 데이터에서는 실제로 보간이 발생하지 않아 전부 동일값(0)으로 확인되어 제외

---


[모델링 시 사용 방법]

X = df[feature_cols]

y = df['y']

--------------------------------

[타겟 설정]
1. 로그변환X

USE_LOG = False:
- y = 평균가격(원)

--

2. 로그변환O

USE_LOG = True:
- y = log1p(평균가격)

--------------------------------

[예측값 복원]

로그 모델일 경우:
pred = np.expm1(pred)

--------------------------------

[결측 처리]

- lag, rolling 생성으로 인한 초기 구간 NaN 발생
- dropna() 후 학습


# 모델링(LightGBM)

## 데이터 업로드

In [6]:
from google.colab import files
uploaded = files.upload()

Saving sample_submission.csv to sample_submission.csv
Saving TEST_00.csv to TEST_00.csv
Saving TEST_01.csv to TEST_01.csv
Saving TEST_02.csv to TEST_02.csv
Saving TEST_03.csv to TEST_03.csv
Saving TEST_04.csv to TEST_04.csv
Saving TEST_05.csv to TEST_05.csv
Saving TEST_06.csv to TEST_06.csv
Saving TEST_07.csv to TEST_07.csv
Saving TEST_08.csv to TEST_08.csv
Saving TEST_09.csv to TEST_09.csv
Saving TEST_10.csv to TEST_10.csv
Saving TEST_11.csv to TEST_11.csv
Saving TEST_12.csv to TEST_12.csv
Saving TEST_13.csv to TEST_13.csv
Saving TEST_14.csv to TEST_14.csv
Saving TEST_15.csv to TEST_15.csv
Saving TEST_16.csv to TEST_16.csv
Saving TEST_17.csv to TEST_17.csv
Saving TEST_18.csv to TEST_18.csv
Saving TEST_19.csv to TEST_19.csv
Saving TEST_20.csv to TEST_20.csv
Saving TEST_21.csv to TEST_21.csv
Saving TEST_22.csv to TEST_22.csv
Saving TEST_23.csv to TEST_23.csv
Saving TEST_24.csv to TEST_24.csv
Saving train_target_only_cleaned.csv to train_target_only_cleaned (1).csv


In [7]:
import os
os.listdir("/content")

['.config',
 'TEST_19.csv',
 'TEST_22.csv',
 'sample_submission.csv',
 'TEST_00.csv',
 'TEST_16.csv',
 'TEST_18.csv',
 'TEST_13.csv',
 'train_target_only_cleaned.csv',
 'train_target_only_cleaned (1).csv',
 'TEST_17.csv',
 'TEST_14.csv',
 'TEST_08.csv',
 'TEST_02.csv',
 'TEST_07.csv',
 'TEST_04.csv',
 'TEST_11.csv',
 'TEST_12.csv',
 'TEST_09.csv',
 'TEST_15.csv',
 'TEST_20.csv',
 'TEST_05.csv',
 'TEST_10.csv',
 'TEST_24.csv',
 'TEST_23.csv',
 'TEST_03.csv',
 'TEST_01.csv',
 'TEST_06.csv',
 'TEST_21.csv',
 'sample_data']

In [8]:
import os

os.makedirs("/content/data/test", exist_ok=True)

In [9]:
import shutil
import os

for f in os.listdir("/content"):
    if f.startswith("TEST_"):
        shutil.move(f"/content/{f}", f"/content/data/test/{f}")
    elif "train" in f:
        shutil.move(f"/content/{f}", "/content/data/train.csv")
    elif "submission" in f:
        shutil.move(f"/content/{f}", "/content/data/sample_submission.csv")

In [10]:
TRAIN_PATH = "/content/data/train.csv"
SAMPLE_SUB_PATH = "/content/data/sample_submission.csv"
TEST_DIR = "/content/data/test"

In [11]:
import glob

print("train:", os.path.exists(TRAIN_PATH))
print("submission:", os.path.exists(SAMPLE_SUB_PATH))

test_files = sorted(glob.glob(TEST_DIR + "/TEST_*.csv"))
print("test 개수:", len(test_files))
print(test_files[:3])

train: True
submission: True
test 개수: 25
['/content/data/test/TEST_00.csv', '/content/data/test/TEST_01.csv', '/content/data/test/TEST_02.csv']


In [12]:
import pandas as pd

sample_sub = pd.read_csv("/content/data/sample_submission.csv")
print(len(sample_sub))

75


In [13]:
for path in test_files:
    df = pd.read_csv(path)
    if len(df) == 0:
        print("문제 있는 파일:", path)

In [15]:
import os
import glob
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

# =========================
# 0. 경로
# =========================
TRAIN_PATH = "/content/data/train.csv"
SAMPLE_SUB_PATH = "/content/data/sample_submission.csv"
TEST_DIR = "/content/data/test"

# =========================
# 1. 데이터 로드
# =========================
train = pd.read_csv(TRAIN_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "TEST_*.csv")))

# =========================
# 2. 시점 파싱 함수
# =========================
def parse_train_time(s):
    s = str(s)
    year = int(s[:4])
    month = int(s[4:6])

    if "상순" in s:
        dekad = 1
    elif "중순" in s:
        dekad = 2
    elif "하순" in s:
        dekad = 3
    else:
        dekad = np.nan

    time_idx = year * 36 + (month - 1) * 3 + dekad
    return year, month, dekad, time_idx

def parse_test_time(s):
    s = str(s)
    if s == "T":
        return 0
    return -int(s.replace("T-", "").replace("순", ""))

def add_time_features_train(df):
    parsed = df["시점"].apply(parse_train_time)
    df["year"] = parsed.apply(lambda x: x[0])
    df["month"] = parsed.apply(lambda x: x[1])
    df["순_num"] = parsed.apply(lambda x: x[2])
    df["time_idx"] = parsed.apply(lambda x: x[3])
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12)
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

def add_future_time_features(base_month, base_dekad, step_ahead):
    total = (base_month - 1) * 3 + base_dekad + step_ahead
    future_month = ((total - 1) // 3) % 12 + 1
    future_dekad = (total - 1) % 3 + 1
    sin_month = np.sin(2 * np.pi * future_month / 12)
    cos_month = np.cos(2 * np.pi * future_month / 12)
    return future_month, future_dekad, sin_month, cos_month

# =========================
# 3. train 피처 생성
# =========================
train = add_time_features_train(train)

group_cols = ["품목명", "품종명", "거래단위", "등급"]
cat_cols = ["품종명", "거래단위", "등급"]
target_col = "평균가격(원)"

train = train.sort_values(group_cols + ["time_idx"]).reset_index(drop=True)

for lag in [1, 2, 3]:
    train[f"lag{lag}"] = train.groupby(group_cols)[target_col].shift(lag)

train["diff1"] = train["lag1"] - train["lag2"]
train["diff2"] = train["lag2"] - train["lag3"]

train["ratio1"] = np.where(train["lag2"] != 0, (train["lag1"] - train["lag2"]) / train["lag2"], 0)
train["ratio2"] = np.where(train["lag3"] != 0, (train["lag2"] - train["lag3"]) / train["lag3"], 0)

train["rolling_mean_3"] = (
    train.groupby(group_cols)[target_col]
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

train_feat = train.dropna(subset=["lag1", "lag2", "lag3", "rolling_mean_3"]).copy()

feature_cols = [
    "품종명", "거래단위", "등급",
    "평년 평균가격(원)",
    "lag1", "lag2", "lag3",
    "diff1", "diff2",
    "ratio1", "ratio2",
    "rolling_mean_3",
    "month", "순_num", "sin_month", "cos_month",
    "time_idx"
]

# =========================
# 4. 품목별 모델 학습
# =========================
models = {}

for item in train_feat["품목명"].unique():
    item_df = train_feat[train_feat["품목명"] == item].copy()

    X = item_df[feature_cols].copy()
    y = item_df[target_col].copy()

    for c in cat_cols:
        X[c] = X[c].astype("category")

    model = LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X, y, categorical_feature=cat_cols)
    models[item] = model

print("학습 완료:", list(models.keys()))

# =========================
# 5. 예측 함수
# =========================
def predict_one_test_file(test_path, models, train, sample_sub):
    test = pd.read_csv(test_path)
    test["rel_idx"] = test["시점"].apply(parse_test_time)

    test_id = os.path.splitext(os.path.basename(test_path))[0]
    item_pred_map = {1: {}, 2: {}, 3: {}}

    for item in test["품목명"].unique():
        item_test = test[test["품목명"] == item].copy()
        pred_list = []

        for keys, grp in item_test.groupby(group_cols):
            grp = grp.sort_values("rel_idx").copy()
            history = grp["평균가격(원)"].tolist()

            if len(history) < 3:
                continue

            t_rows = grp[grp["rel_idx"] == 0]
            if len(t_rows) == 0:
                continue
            last_row = t_rows.iloc[0]

            hist_train = train[
                (train["품목명"] == keys[0]) &
                (train["품종명"] == keys[1]) &
                (train["거래단위"] == keys[2]) &
                (train["등급"] == keys[3])
            ].sort_values("time_idx")

            if len(hist_train) == 0:
                continue

            base_month = hist_train.iloc[-1]["month"]
            base_dekad = hist_train.iloc[-1]["순_num"]
            base_time_idx = hist_train.iloc[-1]["time_idx"]

            current_history = history.copy()
            future_preds = []

            for step in [1, 2, 3]:
                lag1 = current_history[-1]
                lag2 = current_history[-2]
                lag3 = current_history[-3]

                diff1 = lag1 - lag2
                diff2 = lag2 - lag3
                ratio1 = (lag1 - lag2) / lag2 if lag2 != 0 else 0
                ratio2 = (lag2 - lag3) / lag3 if lag3 != 0 else 0
                rolling_mean_3 = np.mean(current_history[-3:])

                future_month, future_dekad, sin_month, cos_month = add_future_time_features(
                    base_month, base_dekad, step
                )

                x_pred = pd.DataFrame([{
                    "품종명": keys[1],
                    "거래단위": keys[2],
                    "등급": keys[3],
                    "평년 평균가격(원)": last_row["평년 평균가격(원)"],
                    "lag1": lag1,
                    "lag2": lag2,
                    "lag3": lag3,
                    "diff1": diff1,
                    "diff2": diff2,
                    "ratio1": ratio1,
                    "ratio2": ratio2,
                    "rolling_mean_3": rolling_mean_3,
                    "month": future_month,
                    "순_num": future_dekad,
                    "sin_month": sin_month,
                    "cos_month": cos_month,
                    "time_idx": base_time_idx + step
                }])

                for c in cat_cols:
                    x_pred[c] = x_pred[c].astype("category")

                pred = models[item].predict(x_pred)[0]
                pred = max(0, pred)

                future_preds.append(pred)
                current_history.append(pred)

            pred_list.append({
                "pred1": future_preds[0],
                "pred2": future_preds[1],
                "pred3": future_preds[2]
            })

        if len(pred_list) > 0:
            item_pred_map[1][item] = np.mean([x["pred1"] for x in pred_list])
            item_pred_map[2][item] = np.mean([x["pred2"] for x in pred_list])
            item_pred_map[3][item] = np.mean([x["pred3"] for x in pred_list])
        else:
            item_pred_map[1][item] = 0
            item_pred_map[2][item] = 0
            item_pred_map[3][item] = 0

    rows = []
    for step in [1, 2, 3]:
        row = {"시점": f"{test_id}+{step}순"}
        for col in sample_sub.columns[1:]:
            row[col] = item_pred_map[step].get(col, 0)
        rows.append(row)

    return pd.DataFrame(rows)

# =========================
# 6. 전체 테스트 예측
# =========================
pred_dfs = []
for test_path in test_files:
    pred_df = predict_one_test_file(test_path, models, train, sample_sub)
    pred_dfs.append(pred_df)

submission = pd.concat(pred_dfs, ignore_index=True)
submission = submission[sample_sub.columns]
submission = sample_sub[["시점"]].merge(submission, on="시점", how="left")

for col in submission.columns[1:]:
    submission[col] = submission[col].fillna(0)
    submission[col] = submission[col].clip(lower=0)

# =========================
# 7. 저장
# =========================
SAVE_PATH = "/content/data/lightgbm_baseline_submission.csv"
submission.to_csv(SAVE_PATH, index=False)

print(submission.head())
print(f"저장 완료: {SAVE_PATH}")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000292 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 507
[LightGBM] [Info] Number of data points in the train set: 137, number of used features: 14
[LightGBM] [Info] Start training from score 34179.918853
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best